In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split

np.random.seed(42)

# Load Optical Recognition of Handwritten Digits Dataset (8x8)
data = load_digits()
X_all = data.images
y_all = data.target

# Convert 0-16 pixel values to bipolar values (-1, +1)
X_bipolar = np.where(X_all > 8, 1, -1)
X_flat = X_bipolar.reshape(len(X_bipolar), 64)

# Use representative examples for each digit
stored_patterns = []
stored_labels = []

for digit in range(10):
    index = np.where(y_all == digit)[0][0]
    stored_patterns.append(X_flat[index])
    stored_labels.append(digit)

stored_patterns = np.array(stored_patterns)
stored_labels = np.array(stored_labels)

# Unique bipolar digit codes (5-bit binary representation)
def digit_to_code(digit):
    return np.array([1 if bit == "1" else -1
                     for bit in format(int(digit), "05b")])

stored_codes = np.array([digit_to_code(d) for d in stored_labels])

# BAM Network
class BAM:
    def __init__(self, X, Y):
        self.X = np.asarray(X, dtype=int)
        self.Y = np.asarray(Y, dtype=int)
        self.W = self.X.T @ self.Y

    def forward(self, x, max_steps=20):
        x = np.asarray(x, dtype=int).copy()
        updates = 0

        for _ in range(max_steps):
            y = np.where(x @ self.W >= 0, 1, -1)
            x_new = np.where(y @ self.W.T >= 0, 1, -1)
            updates += 1

            if np.array_equal(x_new, x):
                return x_new, y, updates

            x = x_new

        y = np.where(x @ self.W >= 0, 1, -1)
        return x, y, updates

    def recall_code(self, x, max_steps=20):
        x = np.asarray(x, dtype=int).copy()
        y_previous = None
        updates = 0

        for _ in range(max_steps):
            y_raw = x @ self.W
            y = np.where(y_raw >= 0, 1, -1)

            x_new = np.where(y @ self.W.T >= 0, 1, -1)
            updates += 1

            if np.array_equal(x_new, x) and (
                y_previous is None or np.array_equal(y, y_previous)
            ):
                return y, x_new, updates

            x = x_new
            y_previous = y.copy()

        y = np.where(x @ self.W >= 0, 1, -1)
        return y, x, updates

    def recall_pattern(self, code, max_steps=20):
        y = np.asarray(code, dtype=int).copy()
        updates = 0

        for _ in range(max_steps):
            x = np.where(y @ self.W.T >= 0, 1, -1)
            y_new = np.where(x @ self.W >= 0, 1, -1)
            updates += 1

            if np.array_equal(y_new, y):
                return x, y_new, updates

            y = y_new

        x = np.where(y @ self.W.T >= 0, 1, -1)
        return x, y, updates

def make_bam(n=10):
    return BAM(stored_patterns[:n], stored_codes[:n])

bam = make_bam(10)

def identify_digit(code, n=10):
    matches = np.where(np.all(stored_codes[:n] == code, axis=1))[0]
    return int(stored_labels[matches[0]]) if len(matches) else "Unknown"

def display_pattern(pattern):
    grid = np.asarray(pattern).reshape(8, 8)
    for row in grid:
        print("".join("██" if pixel == 1 else "  " for pixel in row))

def display_code(code):
    return "".join("1" if bit == 1 else "0" for bit in code)

def pixel_accuracy(a, b):
    return np.mean(np.asarray(a) == np.asarray(b)) * 100

def forward_result(pattern, network):
    code, reconstructed, steps = network.recall_code(pattern)
    return code, reconstructed, steps

# 1. Store Digit-Code Associations
def store_associations():
    print("\nSTORED DIGIT-CODE ASSOCIATIONS")
    print("Forward weight matrix shape:", bam.W.shape)
    print("Transpose weight matrix shape:", bam.W.T.shape)
    print("\nForward Weight Matrix W:")
    print(bam.W)
    print("\nBackward Weight Matrix W.T:")
    print(bam.W.T)

    for i, digit in enumerate(stored_labels):
        print(f"\nDigit: {digit} | Code: {display_code(stored_codes[i])}")
        display_pattern(stored_patterns[i])

# 2. Forward Digit Recall
def forward_recall_challenge():
    digit = int(input("Enter digit to recall (0-9): "))

    if digit not in range(10):
        print("Invalid digit.")
        return

    index = np.where(stored_labels == digit)[0][0]
    pattern = stored_patterns[index]
    code, reconstructed, updates = forward_result(pattern, bam)

    print("\nInput Digit Pattern:")
    display_pattern(pattern)
    print("Calculated Y-layer state:", code)
    print("Stable Output Code:", display_code(code))
    print("Identified Digit:", identify_digit(code))
    print("State Updates:", updates)

# 3. Reverse Digit Reconstruction
def reverse_reconstruction_challenge():
    digit = int(input("Enter digit code to reconstruct (0-9): "))

    if digit not in range(10):
        print("Invalid digit.")
        return

    index = np.where(stored_labels == digit)[0][0]
    code = stored_codes[index]
    reconstructed, stable_code, updates = bam.recall_pattern(code)

    print("\nInput Code:", display_code(code))
    print("Reconstructed Digit Pattern:")
    display_pattern(reconstructed)
    print("Original Digit Pattern:")
    display_pattern(stored_patterns[index])
    print("Reconstruction Accuracy:",
          f"{pixel_accuracy(reconstructed, stored_patterns[index]):.2f}%")
    print("State Updates:", updates)

# 4. Noisy Pattern Association Test
def noisy_pattern_test():
    print("\nNOISY PATTERN ASSOCIATION TEST")
    percentages = [5, 10, 20, 30, 40]

    for digit in range(10):
        index = np.where(stored_labels == digit)[0][0]
        original = stored_patterns[index]
        tolerated = 0

        print(f"\nDigit {digit}")
        for percentage in percentages:
            noisy = original.copy()
            count = max(1, int(len(noisy) * percentage / 100))
            positions = np.random.choice(len(noisy), count, replace=False)
            noisy[positions] *= -1

            code, reconstructed, updates = forward_result(noisy, bam)
            recalled_digit = identify_digit(code)

            correct = recalled_digit == digit
            if correct:
                tolerated = percentage

            print(
                f"Noise: {percentage:2d}% | Recalled Code: "
                f"{display_code(code)} | Identified: {recalled_digit} | "
                f"Correct: {correct} | Updates: {updates}"
            )
            print("Noisy Pattern:")
            display_pattern(noisy)

        print("Maximum Tested Noise Tolerated:", f"{tolerated}%")

# 5. Partial Digit Completion
def partial_digit_completion():
    digit = int(input("Enter digit (0-9): "))

    if digit not in range(10):
        print("Invalid digit.")
        return

    index = np.where(stored_labels == digit)[0][0]
    original = stored_patterns[index]
    incomplete = original.copy()

    print("1. Remove rows")
    print("2. Remove columns")
    print("3. Remove rectangular region")
    option = input("Select option: ")

    if option == "1":
        rows = input("Enter row numbers (0-7), comma separated: ")
        for r in map(int, rows.split(",")):
            if 0 <= r < 8:
                incomplete[r * 8:(r + 1) * 8] = -1

    elif option == "2":
        cols = input("Enter column numbers (0-7), comma separated: ")
        for c in map(int, cols.split(",")):
            if 0 <= c < 8:
                incomplete[c::8] = -1

    elif option == "3":
        r1, r2 = map(int, input("Enter row range (start,end): ").split(","))
        c1, c2 = map(int, input("Enter column range (start,end): ").split(","))
        incomplete.reshape(8, 8)[r1:r2 + 1, c1:c2 + 1] = -1
    else:
        print("Invalid option.")
        return

    code, reconstructed, updates = forward_result(incomplete, bam)

    print("\nIncomplete Pattern:")
    display_pattern(incomplete)
    print("\nReconstructed Pattern:")
    display_pattern(reconstructed)
    print("\nOriginal Pattern:")
    display_pattern(original)
    print("Reconstructed Digit Code:", display_code(code))
    print("Identified Digit:", identify_digit(code))
    print("Pixel Reconstruction Accuracy:",
          f"{pixel_accuracy(reconstructed, original):.2f}%")
    print("State Updates:", updates)

# 6. Bidirectional Recall Consistency
def consistency_analyzer():
    records = []

    for i, digit in enumerate(stored_labels):
        code, reconstructed, updates = forward_result(stored_patterns[i], bam)
        consistency = pixel_accuracy(reconstructed, stored_patterns[i])

        records.append({
            "Digit": digit,
            "Recalled Code": display_code(code),
            "Reconstructed Pixel Accuracy (%)": consistency,
            "Code Correct": identify_digit(code) == digit,
            "State Updates": updates
        })

    df = pd.DataFrame(records)
    print("\nBIDIRECTIONAL RECALL CONSISTENCY")
    print(df.round(2).to_string(index=False))
    print("Average Pixel Consistency:",
          f"{df['Reconstructed Pixel Accuracy (%)'].mean():.2f}%")
    print("Failed Associations:")
    print(df[
        (df["Reconstructed Pixel Accuracy (%)"] < 100) |
        (~df["Code Correct"])
    ].to_string(index=False))

# 7. Association Capacity Challenge
def capacity_challenge():
    rows = []

    for n in [2, 4, 6, 8, 10]:
        network = make_bam(n)
        forward_correct = 0
        backward_scores = []
        consistent = 0

        for i in range(n):
            code, reconstructed, _ = forward_result(
                stored_patterns[i], network
            )
            if np.array_equal(code, stored_codes[i]):
                forward_correct += 1

            reverse_pattern, _, _ = network.recall_pattern(stored_codes[i])
            backward_scores.append(
                pixel_accuracy(reverse_pattern, stored_patterns[i])
            )

            if np.array_equal(reconstructed, stored_patterns[i]):
                consistent += 1

        rows.append({
            "Associations": n,
            "Forward Accuracy (%)": forward_correct / n * 100,
            "Backward Pixel Accuracy (%)": np.mean(backward_scores),
            "Bidirectional Consistency (%)": consistent / n * 100
        })

    df = pd.DataFrame(rows)
    print("\nASSOCIATION CAPACITY CHALLENGE")
    print(df.round(2).to_string(index=False))

    perfect = df[
        (df["Forward Accuracy (%)"] == 100) &
        (df["Backward Pixel Accuracy (%)"] == 100) &
        (df["Bidirectional Consistency (%)"] == 100)
    ]

    if len(perfect):
        print("Largest tested configuration meeting all 100% criteria:",
              int(perfect["Associations"].max()), "associations")
    else:
        print("No tested configuration achieved 100% on every metric.")

# 8. Cross-Association Error Detector
def cross_association_detector():
    error_matrix = np.zeros((10, 10), dtype=int)
    records = []

    for i, digit in enumerate(stored_labels):
        code, reconstructed, updates = forward_result(stored_patterns[i], bam)
        recalled = identify_digit(code)

        if recalled != digit:
            if recalled != "Unknown":
                error_matrix[digit, recalled] += 1

            records.append({
                "Input Digit": digit,
                "Expected Code": display_code(stored_codes[i]),
                "Recalled Code": display_code(code),
                "Incorrectly Associated Digit": recalled,
                "State Updates": updates
            })

    print("\nCROSS-ASSOCIATION ERRORS")
    if records:
        print(pd.DataFrame(records).to_string(index=False))
    else:
        print("No cross-association errors detected.")

    print("\nDigit-to-Digit Error Matrix:")
    print(pd.DataFrame(
        error_matrix,
        index=[f"Actual {i}" for i in range(10)],
        columns=[f"Recalled {i}" for i in range(10)]
    ))

    if error_matrix.sum():
        actual, recalled = np.unravel_index(
            np.argmax(error_matrix), error_matrix.shape
        )
        print(f"Most frequent confusion: {actual} -> {recalled}")
    else:
        print("No confused digit pair found.")

# 9. BAM Associative Memory Report
def associative_memory_report():
    rows = []

    for n in [2, 4, 6, 8, 10]:
        network = make_bam(n)
        forward_correct = 0
        backward_scores = []
        consistency_scores = []
        cross_errors = 0
        noise_tolerance = []

        for i in range(n):
            pattern = stored_patterns[i]
            expected_code = stored_codes[i]
            code, reconstructed, _ = forward_result(pattern, network)

            if np.array_equal(code, expected_code):
                forward_correct += 1
            else:
                cross_errors += 1

            reverse_pattern, _, _ = network.recall_pattern(expected_code)
            backward_scores.append(
                pixel_accuracy(reverse_pattern, pattern)
            )
            consistency_scores.append(
                pixel_accuracy(reconstructed, pattern)
            )

            tolerated = 0
            for percentage in [5, 10, 20, 30, 40]:
                noisy = pattern.copy()
                count = max(1, int(len(noisy) * percentage / 100))
                rng = np.random.default_rng(1000 + i + percentage)
                positions = rng.choice(len(noisy), count, replace=False)
                noisy[positions] *= -1
                recalled_code, _, _ = forward_result(noisy, network)

                if np.array_equal(recalled_code, expected_code):
                    tolerated = percentage

            noise_tolerance.append(tolerated)

        rows.append({
            "Associations": n,
            "Forward Recall (%)": forward_correct / n * 100,
            "Backward Accuracy (%)": np.mean(backward_scores),
            "Noise Tolerance (%)": np.mean(noise_tolerance),
            "Reconstruction (%)": np.mean(backward_scores),
            "Consistency (%)": np.mean(consistency_scores),
            "Cross-Association Errors": cross_errors
        })

    df = pd.DataFrame(rows)
    print("\nBAM ASSOCIATIVE MEMORY REPORT")
    print(df.round(2).to_string(index=False))

    best = df.sort_values(
        ["Forward Recall (%)", "Consistency (%)", "Backward Accuracy (%)"],
        ascending=False
    ).iloc[0]

    print("\nMost Reliable Tested Capacity:",
          int(best["Associations"]), "associations")
    print("Most Robust Tested Configuration:")
    print(best.to_string())

# Menu
def main_menu():
    while True:
        print("\n" + "=" * 65)
        print(" BIDIRECTIONAL ASSOCIATIVE MEMORY (BAM) SYSTEM")
        print("=" * 65)
        print("1. Store Digit-Code Pattern Associations")
        print("2. Forward Digit Recall Challenge")
        print("3. Reverse Digit Reconstruction Challenge")
        print("4. Noisy Pattern Association Test")
        print("5. Partial Digit Completion Challenge")
        print("6. Bidirectional Recall Consistency Analyzer")
        print("7. Association Capacity Challenge")
        print("8. Cross-Association Error Detector")
        print("9. BAM Associative Memory Report")
        print("10. Exit")

        choice = input("\nEnter your choice (1-10): ").strip()

        if choice == "1":
            store_associations()
        elif choice == "2":
            forward_recall_challenge()
        elif choice == "3":
            reverse_reconstruction_challenge()
        elif choice == "4":
            noisy_pattern_test()
        elif choice == "5":
            partial_digit_completion()
        elif choice == "6":
            consistency_analyzer()
        elif choice == "7":
            capacity_challenge()
        elif choice == "8":
            cross_association_detector()
        elif choice == "9":
            associative_memory_report()
        elif choice == "10":
            print("\nPROGRAM FINISHED")
            break
        else:
            print("Invalid choice. Enter a number from 1 to 10.")

main_menu()


 BIDIRECTIONAL ASSOCIATIVE MEMORY (BAM) SYSTEM
1. Store Digit-Code Pattern Associations
2. Forward Digit Recall Challenge
3. Reverse Digit Reconstruction Challenge
4. Noisy Pattern Association Test
5. Partial Digit Completion Challenge
6. Bidirectional Recall Consistency Analyzer
7. Association Capacity Challenge
8. Cross-Association Error Detector
9. BAM Associative Memory Report
10. Exit

Enter your choice (1-10): 10

PROGRAM FINISHED
